<h1 align="center">🦉 Protégé: Ontologies & OWL</h1>

<p align="center"><i>From rules to ontologies: model a domain in OWL, let a description-logic reasoner classify it, and meet the open-world assumption.</i></p>

<p align="center">
  <img src="https://img.shields.io/badge/Course-Knowledge%20Representation%20%26%20Reasoning-6f42c1" />
  <img src="https://img.shields.io/badge/Prot%C3%A9g%C3%A9-OWL%202-8c1515" />
  <img src="https://img.shields.io/badge/Level-Intermediate-yellow" />
</p>

<p align="center">
  <a href="https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/3-protege/protege_ontologies.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</p>

---

## 📌 At a Glance

| | |
|---|---|
| **Topics** | Ontology · OWL 2 · Description Logic · Classes & Properties · Reasoner (HermiT) · SPARQL · Open vs Closed World |
| **Tools** | Protégé (GUI) + `owlready2` & `rdflib` (Python) |
| **Builds on** | First-Order Logic & Prolog: the same domains, now as ontologies |
| **Run time** | About 1 minute |

## 🗂️ Contents
1. [From FOL/Prolog to OWL](#part-1)
2. [Ontology 1: Humans, Dogs & Fish](#part-2)
3. [Ontology 2: University](#part-3)
4. [SPARQL](#part-4)
5. [Open World vs Closed World](#part-5)
6. [Summary](#summary)

---

**Setup** (fresh Colab runtime). `owlready2` calls the HermiT reasoner, which needs Java:

In [1]:
!pip install -q owlready2 rdflib
!java -version 2>&1 | head -1 || apt-get install -y -qq default-jre-headless

In [2]:
import os
from owlready2 import *

os.makedirs('ontologies', exist_ok=True)
print('owlready2 ready')

owlready2 ready

<a id="part-1"></a>

## 🔄 Part 1 · From FOL/Prolog to OWL

> 🎯 **Goal:** See how the FOL/Prolog knowledge maps onto an **ontology**, a shared, machine-readable model of a domain.

**OWL 2** is a W3C language based on **Description Logic** (a decidable fragment of FOL). **Protégé** is the standard GUI editor for it; Python's `owlready2` lets us build and reason over the same files in code.

| English | FOL | Prolog | OWL / Description Logic | Protégé tab |
|---|---|---|---|---|
| Socrates is a human | $Human(socrates)$ | `human(socrates).` | `Human(socrates)` *(ClassAssertion)* | Individuals |
| All humans are mortal | $\forall x\,(Human(x) \rightarrow Mortal(x))$ | `mortal(X) :- human(X).` | $Human \sqsubseteq Mortal$ | Classes → SubClass Of |
| No fish is a mammal | $\forall x\,(Fish(x) \rightarrow \neg Mammal(x))$ | `violation(X) :- fish(X), mammal(X).` | $Fish \sqcap Mammal \sqsubseteq \bot$ | Classes → Disjoint With |
| Every dog has an owner | $\forall x\,(Dog(x) \rightarrow \exists y\,hasOwner(x,y))$ | `has_owner(D) :- dog(D), owns(_, D).` | $Dog \sqsubseteq \exists\, hasOwner.Human$ | Classes → SubClass Of |
| Ali loves AI | $loves(ali, ai)$ | `loves(ali, ai).` | `loves(ali, ai)` *(ObjectPropertyAssertion)* | Individuals |

| FOL / Prolog term | OWL term |
|---|---|
| Constant | **Individual** |
| Unary predicate | **Class** |
| Binary predicate | **Object / Data Property** |
| Rule `head :- body` | **SubClassOf / EquivalentTo** axiom |
| Query | **DL Query** (Protégé) or **SPARQL** |

<a id="part-2"></a>

## 🧬 Part 2 · Ontology 1: Humans, Dogs & Fish

> 🎯 **Goal:** Rebuild the FOL knowledge (humans, dogs, fish) as an OWL ontology and let HermiT derive facts.

### 2.1 Build the ontology
Everything below can be done by clicking in Protégé; here it is in code. File: [`ontologies/animals_domain.owl`](ontologies/animals_domain.owl).

In [3]:
w1 = World()
animals = w1.get_ontology('http://example.org/krr/animals#')

with animals:
    class Animal(Thing): pass
    class Mortal(Thing): pass
    class Human(Animal, Mortal): pass        # Human ⊑ Animal, Human ⊑ Mortal
    class Student(Human): pass
    class Mammal(Animal): pass
    class Fish(Animal): pass
    class Dog(Mammal): pass
    class Topic(Thing): pass
    AllDisjoint([Fish, Mammal])              # no fish is a mammal

    class loves(ObjectProperty):
        domain = [Human]; range = [Topic]
    class hasOwner(ObjectProperty):
        domain = [Dog];   range = [Human]
    class ownerOf(ObjectProperty):
        inverse_property = hasOwner
    Dog.is_a.append(hasOwner.some(Human))    # every dog has an owner

    socrates = Human('socrates'); plato = Human('plato')
    ali = Student('ali'); ai = Topic('ai'); ali.loves = [ai]
    rex = Dog('rex'); rex.hasOwner = [ali]
    bella = Dog('bella')                     # owner exists but is not named
    salmon = Fish('salmon'); dolphin = Mammal('dolphin')
    AllDifferent([socrates, plato, ali, rex, bella, salmon, dolphin])

animals.save('ontologies/animals_domain.owl', format='rdfxml')
print('Classes    :', sorted(c.name for c in animals.classes()))
print('Properties :', sorted(p.name for p in animals.properties()))
print('Individuals:', sorted(i.name for i in animals.individuals()))

Classes    : ['Animal', 'Dog', 'Fish', 'Human', 'Mammal', 'Mortal', 'Student', 'Topic']
Properties : ['hasOwner', 'loves', 'ownerOf']
Individuals: ['ai', 'ali', 'bella', 'dolphin', 'plato', 'rex', 'salmon', 'socrates']

### 2.2 Run the reasoner
We never asserted that Socrates is `Mortal`. The reasoner derives it from $Human \sqsubseteq Mortal$. (In Protégé: *Reasoner → HermiT → Start reasoner*; inferred facts are shown with a yellow background.)

In [4]:
with animals:
    sync_reasoner_hermit(w1, debug=0)

def types(ind):
    """All classes of an individual, direct and inferred."""
    return sorted(c.name for c in ind.INDIRECT_is_a if hasattr(c, 'name') and c.name != 'Thing')

print('socrates is a:', types(socrates))
print('ali      is a:', types(ali))
print('rex      is a:', types(rex))
print()
print('Is socrates Mortal?', Mortal in socrates.INDIRECT_is_a)
print('Is salmon  Mammal? ', Mammal in salmon.INDIRECT_is_a)

socrates is a: ['Animal', 'Human', 'Mortal']
ali      is a: ['Animal', 'Human', 'Mortal', 'Student']
rex      is a: ['Animal', 'Dog', 'Mammal']

Is socrates Mortal? True
Is salmon  Mammal?  False

Two things worth noticing:

- `socrates` is `Mortal`, `Animal`, and `Human`: pure **classification** by subsumption (this is Prolog's `mortal(socrates)` from the First-Order Logic section).
- `salmon` is **not** inferred to be a `Mammal`. The disjointness axiom *forbids* it, so asserting it would make the ontology inconsistent.

### 2.3 Inconsistency detection
Let us break the "no fish is a mammal" rule on purpose, in a throw-away copy of the ontology.

In [5]:
w_bad = World()
bad = w_bad.get_ontology('http://example.org/krr/bad#')
with bad:
    class Animal(Thing): pass
    class Mammal(Animal): pass
    class Fish(Animal): pass
    AllDisjoint([Fish, Mammal])
    nemo = Fish('nemo')
    nemo.is_a.append(Mammal)                 # nemo is a Fish AND a Mammal

try:
    with bad:
        sync_reasoner_hermit(w_bad, debug=0)
    print('Ontology is consistent')
except OwlReadyInconsistentOntologyError:
    print('INCONSISTENT: nemo cannot be both Fish and Mammal (they are disjoint).')

INCONSISTENT: nemo cannot be both Fish and Mammal (they are disjoint).

### 2.4 "Every dog has an owner" and the unnamed owner
`bella` has **no** `hasOwner` fact, yet the axiom $Dog \sqsubseteq \exists hasOwner.Human$ guarantees *some* owner exists. OWL does not need to know who. Compare with Prolog's `has_owner(bella)`, which would be `false` if no `owns(_, bella)` fact were stored.

In [6]:
print('rex   hasOwner:', [o.name for o in rex.hasOwner])
print('bella hasOwner:', [o.name for o in bella.hasOwner], '  (unknown, but an owner must exist)')
print('ali   ownerOf :', [d.name for d in ali.ownerOf], '  (inferred via the inverse property)')

rex   hasOwner: ['ali']
bella hasOwner: []   (unknown, but an owner must exist)
ali   ownerOf : ['rex']   (inferred via the inverse property)

> 💡 **Key Takeaway:** An ontology = classes + properties + axioms. A **reasoner** checks consistency and *classifies* individuals, deriving knowledge that was never written down.

---
<a id="part-3"></a>

## 🎓 Part 3 · Ontology 2: University

> 🎯 **Goal:** Model the university domain (*student, course, enrolled*) with **defined classes** that the reasoner fills automatically. File: [`ontologies/university.owl`](ontologies/university.owl).

```mermaid
classDiagram
    Thing <|-- Person
    Thing <|-- Course
    Thing <|-- Department
    Person <|-- Student
    Person <|-- Professor
    Student <|-- EnrolledStudent
    Student <|-- BusyStudent
    Student --> Course : enrolledIn
    Professor --> Course : teaches
    Person --> Department : memberOf
```

| Axiom | Meaning |
|---|---|
| $\text{enrolledIn}$ domain `Student`, range `Course` | Anything that is enrolled in something **is a** Student |
| $\text{teaches}$ domain `Professor` | Anything that teaches **is a** Professor |
| `EnrolledStudent ≡ Student ⊓ ∃enrolledIn.Course` | Defined class |
| `BusyStudent ≡ Student ⊓ ≥2 enrolledIn.Course` | Defined class (qualified cardinality) |

In [7]:
w2 = World()
uni = w2.get_ontology('http://example.org/krr/university#')

with uni:
    class Person(Thing): pass
    class Student(Person): pass
    class Professor(Person): pass
    class Course(Thing): pass
    class Department(Thing): pass
    AllDisjoint([Student, Professor, Course, Department])

    class enrolledIn(ObjectProperty):
        domain = [Student]; range = [Course]
    class teaches(ObjectProperty):
        domain = [Professor]; range = [Course]
    class taughtBy(ObjectProperty):
        inverse_property = teaches
    class memberOf(ObjectProperty):
        domain = [Person]; range = [Department]
    class hasName(DataProperty, FunctionalProperty):
        range = [str]
    class hasCredits(DataProperty, FunctionalProperty):
        domain = [Course]; range = [int]

    # Defined classes: membership is *computed* by the reasoner
    class EnrolledStudent(Student): pass
    EnrolledStudent.equivalent_to = [Student & enrolledIn.some(Course)]
    class BusyStudent(Student): pass
    BusyStudent.equivalent_to = [Student & enrolledIn.min(2, Course)]

    cs = Department('computer_science')
    krr = Course('krr_cs331'); ai = Course('intro_ai')
    krr.hasCredits = 3; ai.hasCredits = 3
    ali = Student('ali'); sara = Student('sara'); bilal = Student('bilal')
    ali.enrolledIn = [krr]
    sara.enrolledIn = [krr, ai]
    fatima = Professor('dr_fatima'); fatima.teaches = [krr]; fatima.memberOf = [cs]
    zoya = Thing('zoya'); zoya.enrolledIn = [ai]     # NO type given
    khan = Thing('dr_khan'); khan.teaches = [ai]     # NO type given
    for i, n in [(ali,'Ali'), (sara,'Sara'), (bilal,'Bilal'),
                 (fatima,'Dr. Fatima'), (zoya,'Zoya'), (khan,'Dr. Khan')]:
        i.hasName = n
    AllDifferent([krr, ai]); AllDifferent([ali, sara, bilal, fatima, zoya, khan])

uni.save('ontologies/university.owl', format='rdfxml')
print('Saved university.owl')

Saved university.owl

### 3.1 Before reasoning: what we *asserted*

In [8]:
for ind in [ali, sara, bilal, zoya, khan]:
    print(f'{ind.name:6} asserted types: {[c.name for c in ind.is_a]}')

ali    asserted types: ['Student']
sara   asserted types: ['Student']
bilal  asserted types: ['Student']
zoya   asserted types: ['Thing']
dr_khan asserted types: ['Thing']

### 3.2 After reasoning: what HermiT *inferred*

In [9]:
with uni:
    sync_reasoner_hermit(w2, debug=0)

def types(ind):
    return sorted(c.name for c in ind.INDIRECT_is_a if hasattr(c, 'name') and c.name != 'Thing')

for ind in [ali, sara, bilal, zoya, khan]:
    print(f'{ind.name:6} -> {types(ind)}')

ali    -> ['EnrolledStudent', 'Person', 'Student']
sara   -> ['BusyStudent', 'EnrolledStudent', 'Person', 'Student']
bilal  -> ['Person', 'Student']
zoya   -> ['EnrolledStudent', 'Person', 'Student']
dr_khan -> ['Person', 'Professor']

Read the result:

- `zoya` and `dr_khan` had **no type**, yet they are now `Student` and `Professor`: the *domain* axioms of `enrolledIn` and `teaches` let the reasoner deduce it.
- `ali`, `sara`, `zoya` are `EnrolledStudent` (they have at least one `enrolledIn`).
- Only `sara` is a `BusyStudent` (two *different* courses; we declared them `AllDifferent`).
- `bilal` is just a `Student`. Section 5 explains why.

---
<a id="part-4"></a>

## 🔎 Part 4 · Querying with SPARQL

> 🎯 **Goal:** Ask questions of the ontology, including over *inferred* facts. (Protégé has two query tools: **DL Query** for class expressions, and the **SPARQL Query** tab.)

**DL Query** examples you can paste into Protégé (*Window → Tabs → DL Query*):

```
Student and enrolledIn some Course
Student and enrolledIn min 2 Course
Professor and teaches value krr_cs331
Dog and hasOwner value ali
```

In [10]:
PREFIX = ("PREFIX u: <http://example.org/krr/university#>\n"
          "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n")
# `a/rdfs:subClassOf*` = "has type C, or a type that is a subclass of C"

def sparql(q):
    return [row for row in w2.sparql(PREFIX + q)]

print('Enrolled students (inferred class):')
for (s,) in sparql("SELECT DISTINCT ?s WHERE { ?s a/rdfs:subClassOf* u:EnrolledStudent }"):
    print('  ', s.name)

print('Who is a Student? (includes inferred zoya):')
for (s,) in sparql("SELECT DISTINCT ?s WHERE { ?s a/rdfs:subClassOf* u:Student }"):
    print('  ', s.name)

print('Who teaches what:')
for (p, c) in sparql("SELECT ?p ?c WHERE { ?p u:teaches ?c }"):
    print(f'   {p.name} teaches {c.name}')

print('Names of people enrolled in krr_cs331:')
for (n,) in sparql("SELECT ?n WHERE { ?s u:enrolledIn u:krr_cs331 . ?s u:hasName ?n }"):
    print('  ', n)

Enrolled students (inferred class):
   ali
   sara
   zoya
Who is a Student? (includes inferred zoya):
   ali
   sara
   bilal
   zoya
Who teaches what:
   dr_fatima teaches krr_cs331
   dr_khan teaches intro_ai
Names of people enrolled in krr_cs331:
   Ali
   Sara

> 💡 **Key Takeaway:** Prolog answers queries by *proof search*; with OWL we first **classify** with a reasoner, then query with **SPARQL** or **DL Query**.

---
<a id="part-5"></a>

## 🌍 Part 5 · Open World vs Closed World

> 🎯 **Goal:** Understand the most important difference between Prolog and OWL.

| | **Prolog** | **OWL** |
|---|---|---|
| Assumption | **Closed World** (CWA): what is not provable is *false* | **Open World** (OWA): what is not stated is *unknown* |
| Negation | Negation as failure `\+` | Classical $\neg$ |
| Unique names | Yes: `ali` and `bilal` are different | No: needs `AllDifferent` |

Earlier, the Prolog rule `idle(S) :- student(S), \+ enrolled(S, _).` returned **bilal**. Let us define the same idea in OWL and see.

In [11]:
with uni:
    class IdleStudent(Student): pass
    IdleStudent.equivalent_to = [Student & Not(enrolledIn.some(Course))]

with uni:
    sync_reasoner_hermit(w2, debug=0)

print('Prolog (closed world):  idle(S)  ->  [bilal]')
print('OWL    (open world)  :  IdleStudent members ->', [i.name for i in IdleStudent.instances()])
print('bilal types          :', types(bilal))

Prolog (closed world):  idle(S)  ->  [bilal]
OWL    (open world)  :  IdleStudent members -> []
bilal types          : ['Person', 'Student']

OWL does **not** conclude that `bilal` is idle: he has no *stated* enrolment, but a missing fact only means *unknown*; he might be enrolled somewhere we have not recorded. To get Prolog's behaviour we must **close** the world for him explicitly, with a cardinality restriction ("bilal is enrolled in at most 0 courses"):

In [12]:
with uni:
    bilal.is_a.append(enrolledIn.max(0, Course))   # explicit closure
    sync_reasoner_hermit(w2, debug=0)

print('After closing the world for bilal:')
print('IdleStudent members ->', [i.name for i in IdleStudent.instances()])
print('bilal types         ->', types(bilal))

After closing the world for bilal:
IdleStudent members -> ['bilal']
bilal types         -> ['IdleStudent', 'Person', 'Student']

> 💡 **Key Takeaway:** Prolog **assumes** completeness (closed world); OWL **admits** incompleteness (open world). Pick Prolog for databases with complete data and rules; pick OWL for shared, evolving, incomplete knowledge on the web.

---
<a id="summary"></a>
## ✅ Summary

| Part | Topic | What you learned |
|---|---|---|
| 1 | FOL/Prolog → OWL | Constant→Individual, predicate→Class/Property, rule→axiom |
| 2 | Animals ontology | Subsumption, disjointness, existential restriction, inconsistency detection |
| 3 | University ontology | Domain/range inference, defined classes, cardinality |
| 4 | SPARQL & DL Query | Querying asserted and inferred knowledge |
| 5 | Open vs Closed World | Why `\+` in Prolog ≠ ¬ in OWL |

## 🛠️ Do it in Protégé
Open `ontologies/university.owl` in [Protégé](https://protege.stanford.edu/): inspect *Classes*, *Object Properties* and *Individuals*, start **HermiT**, then try the DL queries from Part 4. A step-by-step guide is in the [main README](../README.md#-protégé-guide).

<p align="center"><i>⬅️ Back to Prolog · Next: Project ➡️</i></p>